# 2 · Benchmark and post-train SmolVLA on my hand demos (GPU runtime)

Notebook 1 turned my phone clips into successful robot replays, saved as demos on Drive.
This notebook asks: **does SmolVLA get better at the task after training on them?**

```
baseline benchmark ─► build dataset from demos ─► post-train ─► benchmark again ─► compare
```

Task: LIBERO `libero_object` task 0, *"pick up the alphabet soup and place it in the basket"*.
Test = layouts 0–24 (25 episodes); the demos come from layouts 25+, so the test is unseen.
Both models are tested with the same script and settings (`scripts/vla_benchmark.py`).

Runtime → Change runtime type → **GPU** before running. Each step skips itself if its result already exists,
so after a disconnect just run everything again from the top.

**Examiners:** with the default settings (`MODE = "examiner"`, `RERUN = False`) the notebook shows my saved
results from the shared folder and only computes what is missing, writing to `/content/out`.

## 0 · Setup (≈5 min, once per session)

In [ ]:
from google.colab import drive; drive.mount('/content/drive')
import os
if not os.path.exists("/content/lerobot"):
    !git clone -q https://github.com/huggingface/lerobot.git /content/lerobot
!pip install -q -e "/content/lerobot[smolvla,libero]"
if not os.path.exists("/content/repo"):
    !git clone -q https://github.com/Suhail-Halaby/humanoidRLassignment.git /content/repo
!cd /content/repo && git pull -q && git log --oneline -1
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

## 1 · Settings

In [ ]:
# Who is running this notebook?
#   "examiner": read my shared Drive folder (shortcut called humanoid, see README "Data access");
#               anything you produce is written to /content/out
#   "author":   my own Drive folder, read and write
MODE = "examiner"
RERUN = False   # False: show my saved results where they exist (no GPU hours needed)
                # True:  run every step yourself, into /content/out
if MODE == "author":
    DATA = OUT = "/content/drive/MyDrive/humanoid"
else:
    DATA, OUT = "/content/drive/MyDrive/humanoid", "/content/out"
    os.makedirs(OUT, exist_ok=True)
assert os.path.exists(DATA), f"{DATA} not found: add the shared folder to your Drive (README, Data access)"

def where(name, done_marker):
    # my saved result if it is complete (and RERUN is off), otherwise a fresh folder in OUT
    if not RERUN and os.path.exists(f"{DATA}/{name}/{done_marker}"):
        return f"{DATA}/{name}"
    return f"{OUT}/{name}"

DEMOS = f"{DATA}/demos"                                               # successful replays (notebook 1)
DATASET = where("dataset_hand2panda", "meta")                         # LeRobot dataset built from DEMOS
TRAIN = where("train_hand2panda", "checkpoints/last/pretrained_model")  # training output (~1.2 GB / checkpoint)
EVAL_BASE = where("eval_baseline_25", "eval_info.json")               # benchmark of the pretrained model
EVAL_POST = where("eval_posttrained_25", "eval_info.json")            # benchmark of the post-trained model
MODEL_POST = f"{TRAIN}/checkpoints/last/pretrained_model"
for k in ["DEMOS", "DATASET", "TRAIN", "EVAL_BASE", "EVAL_POST"]:
    print(f"{k:10s} {globals()[k]}")

STEPS = 2000                                # training updates (see scripts/train_vla.py)
EPISODES = 25                               # test episodes = layouts 0..24

import json, re, glob
def result(folder):
    # headline numbers from LeRobot's eval_info.json: successes, interval, s/episode, per-layout success
    o = json.load(open(f"{folder}/eval_info.json"))
    ov = o["overall"]
    per = o["per_task"][0]["metrics"]["successes"]
    return ov["n_success"], ov["n_episodes"], ov["pc_success_ci95"], ov["eval_ep_s"], [bool(s) for s in per]

## 2 · Baseline: the pretrained SmolVLA (≈35 min)
`HuggingFaceVLA/smolvla_libero`, never shown my data. Skipped if already done.

In [ ]:
if os.path.exists(f"{EVAL_BASE}/eval_info.json"):
    print("baseline already done")
else:
    !rm -rf {EVAL_BASE}
    !python /content/repo/scripts/vla_benchmark.py --episodes {EPISODES} --out {EVAL_BASE}
n, N, ci, s, _ = result(EVAL_BASE)
print(f"BASELINE: {n}/{N}  (95% interval {ci[0]:.0f}-{ci[1]:.0f}%)  {s:.0f} s/episode")

## 3 · Build the training dataset from the demos (≈5 min)
Every file in `DEMOS` is a replay that **succeeded** (notebook 1 only saves successes).

In [ ]:
demo_files = sorted(glob.glob(f"{DEMOS}/*.npz"))
clips = sorted({os.path.basename(f).split("_layout")[0] for f in demo_files})
print(f"{len(demo_files)} demos from {len(clips)} clips: {clips}")
if os.path.exists(f"{DATASET}/meta"):
    print("dataset already built")
else:
    !rm -rf {DATASET}
    !python /content/repo/scripts/to_lerobot.py --demos {DEMOS} --out {DATASET}

## 4 · Post-train (≈1 h)
Starts from the same pretrained model and continues training on my demos (imitation learning:
predict the demo's action from what the robot saw, measure the error, adjust the weights).
Only SmolVLA's action expert is trained; settings and reasons are in `scripts/train_vla.py`.
After the first ~100 steps, check the speed in the log; if 2000 steps would take far more than an hour,
stop the cell, set `STEPS = 1000` in Settings, delete `TRAIN` and run again.

In [ ]:
if os.path.exists(MODEL_POST):
    print("already trained")
else:
    !rm -rf {TRAIN}
    !python /content/repo/scripts/train_vla.py --dataset {DATASET} --out {TRAIN} --steps {STEPS}

In [ ]:
# training loss: should fall quickly, then flatten
import matplotlib.pyplot as plt
def num(x):  # LeRobot prints 1000 as "1K"
    return float(x[:-1]) * 1000 if x.endswith("K") else float(x)
pts = [(num(m.group(1)), float(m.group(2)))
       for m in re.finditer(r"step:(\S+) .*?loss:([\d.]+)", open(f"{TRAIN}_log.txt").read())]
if pts:
    plt.figure(figsize=(6, 3)); plt.plot(*zip(*pts), marker=".")
    plt.xlabel("training step"); plt.ylabel("loss"); plt.title("post-training loss"); plt.grid(alpha=.3)
    plt.savefig(f"{OUT}/train_loss.png", dpi=120, bbox_inches="tight"); plt.show()
else:
    print("no loss lines found in", f"{TRAIN}_log.txt")

## 5 · Benchmark the post-trained model (≈35 min)
Same script, same 25 layouts; only the model changes.

In [ ]:
if os.path.exists(f"{EVAL_POST}/eval_info.json"):
    print("post-trained benchmark already done")
else:
    !rm -rf {EVAL_POST}
    !python /content/repo/scripts/vla_benchmark.py --policy {MODEL_POST} --episodes {EPISODES} --out {EVAL_POST}

## 6 · Compare

In [ ]:
if not os.path.exists(f"{EVAL_POST}/eval_info.json"):
    raise SystemExit("post-trained benchmark not available yet (section 5) - only the baseline above can be shown")
b = result(EVAL_BASE); p = result(EVAL_POST)
print(f"{'':14s}{'success':>10s}{'95% interval':>16s}{'s/episode':>11s}")
for name, (n, N, ci, s, _) in [("baseline", b), ("post-trained", p)]:
    print(f"{name:14s}{f'{n}/{N}':>10s}{f'{ci[0]:.0f}-{ci[1]:.0f}%':>16s}{s:>11.0f}")

# layout by layout: where did post-training help or hurt?
gained = [i for i, (x, y) in enumerate(zip(b[4], p[4])) if y and not x]
lost   = [i for i, (x, y) in enumerate(zip(b[4], p[4])) if x and not y]
print("\nlayouts newly solved:", gained or "none")
print("layouts newly failed:", lost or "none")

plt.figure(figsize=(9, 1.8))
plt.imshow([b[4], p[4]], cmap="RdYlGn", vmin=0, vmax=1, aspect="auto")
plt.yticks([0, 1], ["baseline", "post-trained"]); plt.xticks(range(len(b[4])))
plt.xlabel("test layout (green = success)"); plt.title("per-layout results")
plt.savefig(f"{OUT}/per_layout.png", dpi=120, bbox_inches="tight"); plt.show()

**Reading it honestly:** with 25 episodes per model the 95% intervals are wide and usually overlap.
A difference of 1–3 episodes is within chance; report the numbers with their intervals either way.
The per-layout row shows *where* the models differ — useful to look at the matching videos.

In [ ]:
# watch one episode of each model (videos saved by LeRobot)
import mediapy
for name, folder in [("baseline", EVAL_BASE), ("post-trained", EVAL_POST)]:
    if not os.path.exists(folder): continue
    vids = sorted(glob.glob(f"{folder}/videos/**/*.mp4", recursive=True))   # works from any Drive path
    if vids:
        print(name, "-", os.path.basename(vids[0]))
        mediapy.show_video(mediapy.read_video(vids[0]), fps=20, width=300)